In [0]:
WITH CREDIT_ASSESSMENT AS
(
SELECT  _id, `_created_by.Name`, 
        date_format(`_created_at.value`,'yyyy-MM-dd') AS `_created_at.value`,  
        `_created_at.timezone`, 
        date_format(`_submitted_at.value`,'yyyy-MM-dd') AS `_submitted_at.value`, 
        `_modified_by.Name`, 
        date_format(`_modified_at.value`,'yyyy-MM-dd') AS `_modified_at.value`, 
        _current_assigned_to, _current_step, 
        date_format(`_completed_at.value`,'yyyy-MM-dd') AS `_completed_at.value`,  
        _status, _progress, Name, 
        TRIM(split_part(Name, '|', -2 )) AS No_Credit_Assessment_KF,
        TRIM(regexp_replace(CASE WHEN regexp_instr(Group_Name__Hid, '[0-9]') = '1' AND regexp_count(group_name__hid, '[0-9]') > 4 
        THEN REPLACE(Group_Name__Hid, substring(Group_Name__Hid, 1, regexp_instr(group_name__hid, '[A-Z]') -1), '') 
        WHEN regexp_instr(Group_Name__Hid, '[0-9]') > '1' AND regexp_count(group_name__hid, '[0-9]') > 4 
        THEN REPLACE(Group_Name__Hid, substring(Group_Name__Hid, regexp_instr(Group_Name__Hid, '[0-9]') -1, len(Group_Name__Hid)), '') 
        ELSE Group_Name__Hid END, '[-|/]', '')) AS Group_Name__Hid,
        regexp_instr(Group_Name__Hid, '[0-9]') AS index_regexp, 
        TRIM(regexp_replace(split_part(split_part(Customer_Number, ',', 1), ':', 2), '[-"A-Z]', '')) AS Customer_Number,
        Business_Segment_1, Region, CountryEntity, Type_of_request, Payment_Terms, Updated_Payment_Terms, 
        format_number(Current_credit_limit, 2) AS Current_credit_limit, 
        format_number(Requested_credit_limit, 2) AS Requested_credit_limit, 
        format_number(Key_Money_Amount, 2) AS Key_Money_Amount, 
        format_number(Credit_Limit_Total_Hidden, 2) AS Credit_Limit_Total_Hidden, 
        Select_Security_Type, Total_Security_Offered, Projected_Volume, Projected_Gross_Margin_USD, Payout_Years, TAT_calculation, Apprehensions, 
        date_format(Last_Reviewed_Date,'yyyy-MM-dd') AS Last_Reviewed_Date, 
        Type_of_review, Risk_Profile, Financial_Strength, Financial_statement_analysis, Flow_ID_for_Link

FROM kissflow.default.databricks_output_2025_08_29 
WHERE `_created_at.value` >= '2025-01-01' 
)

SELECT  count_if(customer_number = '') AS EMPTY_CUST_NO, 
        count_if(customer_number <> '') AS NOT_EMPTY_CUST_NO, 
        count_if(group_name__hid = '') AS EMPTY_GROUP_NAME_HID, 
        count_if(group_name__hid <> '') AS NOT_EMPTY_GROUP_NAME_HID,
        COUNT(*) AS TOTAL
FROM CREDIT_ASSESSMENT; 

----------------------------------------------------------------------
SELECT Group_Name__Hid_Cleaned
FROM CREDIT_ASSESSMENT
WHERE regexp_instr(Group_Name__Hid_Cleaned, '[-/|]') = len(Group_Name__Hid_Cleaned) -- VALIDAR SI ÚLTIMO ÍNDICE DE UNA VARIABLE/STRING ES CARACTER
AND Group_Name__Hid_Cleaned <> '';                                                  -- ESPECIAL.

----------------------------------------------------------------------
--JOIN WITH LEVENSHTEIN FUNCTION: 
SELECT  
        CA.Group_Name__Hid,
        C.Name, 
        levenshtein(CA.Group_Name__Hid, C.Name) AS LEVENSHTEIN_SCORE
        --C.SearchName, 
        --CA.Customer_Number,
        --C.UID, 
        --C.UIDName
        --C.Name2,
        --C.No_  
FROM CREDIT_ASSESSMENT AS CA
LEFT JOIN CUSTOMER AS C
ON levenshtein(CA.Group_Name__Hid, C.Name) <= 2 
WHERE CA.Group_Name__Hid IS NOT NULL
AND CA.Group_Name__Hid != ''
AND C.Name IS NOT NULL
ORDER BY levenshtein_score DESC;

-----------------------------------------------------------------------------
--QUERY UP TO DATE
WITH CREDIT_ASSESSMENT AS
(
SELECT  _id, `_created_by.Name`, 
        `_created_at.value`,  
        `_created_at.timezone`, 
        `_submitted_at.value`, 
        `_modified_by.Name`, 
        `_modified_at.value`,
        _current_assigned_to, _current_step, 
        `_completed_at.value`,  
        _status, _progress, Name, 
        CASE WHEN Requested_credit_limit <= 100000 THEN 'LOCAL'
        WHEN Requested_credit_limit > 100000 AND Requested_credit_limit <= 1000000 THEN 'REGIONAL'
        WHEN Requested_credit_limit > 1000000 THEN 'GLOBAL' ELSE NULL END AS DOA_By_Requested_CL,
        TRIM(split_part(Name, '|', -2 )) AS No_Credit_Assessment_KF,
        Group_Name__Hid, 
        TRIM(regexp_replace(CASE WHEN regexp_instr(Group_Name__Hid, '[0-9]') = '1' AND regexp_count(group_name__hid, '[0-9]') > 4 
        THEN REPLACE(Group_Name__Hid, substring(Group_Name__Hid, 1, regexp_instr(group_name__hid, '[A-Z]') -1), '') 
        WHEN regexp_instr(Group_Name__Hid, '[0-9]') > '1' AND regexp_count(group_name__hid, '[0-9]') > 4 
        THEN REPLACE(Group_Name__Hid, substring(Group_Name__Hid, regexp_instr(Group_Name__Hid, '[0-9]') -1, len(Group_Name__Hid)), '') 
        ELSE Group_Name__Hid END, '[-|/]', '')) AS Group_Name__Hid_Cleaned,
        regexp_instr(Group_Name__Hid, '[0-9]') AS index_regexp, 
        TRIM(regexp_replace(split_part(split_part(Customer_Number, ',', 1), ':', 2), '[-"A-Z]', '')) AS Customer_Number,
        Business_Segment_1, Region, CountryEntity, Type_of_request, Payment_Terms, Updated_Payment_Terms, 
        cast(Current_credit_limit AS double) AS Current_credit_limit,
        cast(Requested_credit_limit AS double) AS Requested_credit_limit, 
        cast(Key_Money_Amount AS double) AS Key_Money_Amount, 
        cast(Credit_Limit_Total_Hidden AS double) AS Credit_Limit_Total_Hidden,
        Select_Security_Type, Total_Security_Offered, Security_Type_Hidden, Projected_Volume, Projected_Gross_Margin_USD, Payout_Years, TAT_calculation, Apprehensions, 
        date_format(Last_Reviewed_Date,'yyyy-MM-dd') AS Last_Reviewed_Date, 
        Type_of_review, Risk_Profile, Financial_Strength, Financial_statement_analysis, Flow_ID_for_Link

FROM kissflow.default.databricks_output_2025_08_29 
WHERE `_created_at.value` >= '2025-01-01' 
),

NAV_CUSTOMER AS 
(SELECT Name, 
        UIDName,
        ROW_NUMBER() OVER(PARTITION BY Name ORDER BY Name) AS RN
FROM prodsqldwh.adls.nav_customer_dim
WHERE CompanyCode IN ('COS', 'EPA', 'ELS', 'EBE', 'ENC', 'PEG', 'EGU', 'HON', 'HOR', 'EHO', 'PCA', 'PCP')
AND IsDeleted = False
AND UIDName <> 'null'
), 

GROUP_MASTER AS 
(SELECT CompanyCode,
        CAST(CustomerNumber AS STRING) AS CustomerNumber,
        GroupName,
        AddedOn,
        ModifiedOn,
        ROW_NUMBER() OVER(PARTITION BY CustomerNumber ORDER BY CustomerNumber DESC) AS RN
FROM prodsqldwh.adls.customergroupmaster
), 

CA_MERGE_NC AS
(
SELECT CA.*, 
        NC.UIDName
FROM CREDIT_ASSESSMENT CA
LEFT JOIN NAV_CUSTOMER NC
 ON CA.Group_Name__Hid_Cleaned = NC.Name
WHERE NC.RN = 1 OR NC.RN IS NULL
)

SELECT M.*, 
        GM.GroupName, 
        CASE WHEN UIDName IS NULL AND GroupName IS NOT NULL THEN GroupName
        WHEN UIDName IS NOT NULL AND GroupName IS NULL THEN UIDName
        WHEN UIDName IS NOT NULL AND GroupName IS NOT NULL THEN UIDName 
        ELSE Group_Name__Hid_Cleaned END AS UIDName_consolidated
FROM CA_MERGE_NC M
LEFT JOIN GROUP_MASTER GM
ON M.customer_number= GM.customernumber
WHERE GM.RN = 1 OR GM.RN IS NULL
;


SELECT * 
FROM prodsqldwh.adls.nav_customer_dim;
